In [ ]:
import numpy as np
import pandas as pd

Data=pd.read_csv("withGDP_IRD_with_tone_dataset.csv")
# Your already-forward-filled monthly dataset
df = Data.copy()

# Columns to jitter (all forward-filled sentiment columns)
filled_cols = [c for c in df.columns if c.endswith("_filled")]

# Reproducibility
rng = np.random.default_rng(42)

def add_jitter_to_plateaus(series: pd.Series, rng, frac: float = 0.25, eps_floor: float = 1e-6):
    """
    Adds mean-zero Gaussian jitter ONLY to plateau runs (consecutive identical values),
    excluding the first value of each run. Jitter scale is tied to typical non-zero
    month-to-month changes so it stays small.
    """
    s = series.astype(float).copy()

    # Identify plateau positions: same as previous month
    plateau = s.eq(s.shift(1))

    # Estimate a sensible noise scale from non-zero first differences
    d = s.diff()
    nonzero_d = d[(d.abs() > 0) & d.notna()]

    # If all diffs are zero (rare), fall back to overall std or a tiny floor
    base_scale = nonzero_d.std()
    if pd.isna(base_scale) or base_scale == 0:
        base_scale = s.std()
    if pd.isna(base_scale) or base_scale == 0:
        base_scale = eps_floor

    sigma = max(frac * base_scale, eps_floor)

    # Add noise only on plateau months (not the first in each constant run)
    noise = rng.normal(loc=0.0, scale=sigma, size=int(plateau.sum()))
    s.loc[plateau] = s.loc[plateau] + noise

    return s

# Apply jitter; create new columns (keeps originals intact)
for col in filled_cols:
    df[col.replace("_filled", "_jittered")] = add_jitter_to_plateaus(df[col], rng=rng, frac=0.25)

# Optional: if these are probabilities (0–1), clip jittered values to [0, 1]
prob_like = ["tone", "pos", "neg", "neu"]
for col in filled_cols:
    new_col = col.replace("_filled", "_jittered")
    if any(k in col.lower() for k in prob_like):
        df[new_col] = df[new_col].clip(0, 1)

# If you want the jittered columns to replace the filled ones:
# for col in filled_cols:
#     df[col] = df[col.replace("_filled", "_jittered")]

withGDP_IRD_with_tone_dataset_jittered = df

In [ ]:
withGDP_IRD_with_tone_dataset_jittered.to_csv("withGDP_IRD_with_tone_dataset_jittered.csv", index=False)